# PDF OCR tiếng Việt - Tesseract (thay thế MinerU OCR)

Pipeline dùng **Tesseract** cho OCR tiếng Việt thay vì PaddleOCR trong MinerU.

Tesseract `vie` thường nhận diện tiếng Việt tốt hơn model latin của MinerU.

## 1. Cài đặt

In [ ]:
# Cài Tesseract + poppler (cho pdf2image) + data tiếng Việt
!apt-get update -qq && apt-get install -y -qq tesseract-ocr tesseract-ocr-vie poppler-utils > /dev/null
!pip install -q pdf2image pytesseract Pillow

In [ ]:
!tesseract --version
!tesseract --list-langs | grep vie

## 2. Upload PDF

In [ ]:
from google.colab import files
uploaded = files.upload()

import shutil
import os
os.makedirs("/content/input", exist_ok=True)
os.makedirs("/content/output", exist_ok=True)

for filename in uploaded.keys():
    shutil.move(filename, f"/content/input/{filename}")
    print(f"Đã upload: {filename}")

## 3. OCR PDF bằng Tesseract tiếng Việt

In [ ]:
from pdf2image import convert_from_path
import pytesseract
import os

pdf_files = [f for f in os.listdir("/content/input") if f.lower().endswith(".pdf")]
if not pdf_files:
    raise FileNotFoundError("Chưa có file PDF. Hãy upload ở cell trên.")
pdf_path = f"/content/input/{pdf_files[0]}"
print(f"Đang xử lý: {pdf_path}")

# Chuyển PDF sang ảnh (200 DPI cho chất lượng tốt)
images = convert_from_path(pdf_path, dpi=200)
print(f"Số trang: {len(images)}")

In [ ]:
# OCR từng trang với Tesseract tiếng Việt
all_text = []
for i, img in enumerate(images):
    text = pytesseract.image_to_string(img, lang="vie")
    all_text.append(f"## Trang {i+1}\n\n{text}")
    print(f"Trang {i+1}/{len(images)} xong")

full_md = "\n\n---\n\n".join(all_text)

## 4. Xem và tải kết quả

In [ ]:
print(full_md[:3000])

In [ ]:
# Lưu file Markdown
base_name = os.path.splitext(os.path.basename(pdf_path))[0]
output_path = f"/content/output/{base_name}_tesseract_vi.md"
with open(output_path, "w", encoding="utf-8") as f:
    f.write(full_md)
print(f"Đã lưu: {output_path}")

In [ ]:
# Tải về máy
from google.colab import files
files.download(output_path)
print("Đã tải file!")